# Phase 5 L1/L2/L3 - Resist Levels

Compare Level 0 threshold resist, Level 1 Gaussian blur, Level 2 depth-resolved dose, and Level 3 stochastic LWR.

In [ ]:
from pathlib import Path
import sys

repo = Path.cwd().resolve()
if not (repo / "src").exists() and (repo.parent / "src").exists():
    repo = repo.parent
if str(repo) not in sys.path:
    sys.path.insert(0, str(repo))

from src import constants as C
from src.dof import focus_drilling_average
from src.mask import MaskGrid, kirchhoff_mask, line_space_pattern
from src.metrics import critical_dimension, edge_positions, mean_absolute_epe
from src.pupil import PupilSpec
from src.resist_blur import blurred_threshold_resist, blur_dose_sweep, calibrate_blur_lwr_proxy, gaussian_blur, transition_width
from src.resist_depth import calibrate_sidewall_angle_proxy, focus_depth_resolved_resist, sidewall_angle_proxy, top_bottom_dose_asymmetry
from src.resist_stochastic import StochasticResistParams, calibrate_stochastic_lwr_budget, monte_carlo_convergence_gate, monte_carlo_lwr_curve
from src.resist_threshold import threshold_resist

In [ ]:
pitch_m = 80e-9
grid = MaskGrid(nx=512, ny=64, pixel_size=1e-9)
pattern = line_space_pattern(grid, pitch_m=pitch_m, duty_cycle=0.5)
mask = kirchhoff_mask(pattern)

aerial, wafer = focus_drilling_average(
    mask,
    grid,
    [0.0],
    pupil_spec=PupilSpec(grid_size=512, na=C.NA_HIGH, obscuration_ratio=0.0),
    anamorphic=False,
)

printed_l0 = threshold_resist(aerial, dose=1.0, threshold=0.2)
printed_l1 = blurred_threshold_resist(aerial, wafer.pixel_x_m, sigma_m=2e-9, dose=1.0, threshold=0.2)

In [ ]:
cy = wafer.ny // 2
target_clear = pattern[cy, :] == 0.0
target_cd = critical_dimension(target_clear, grid.pixel_size)
target_edges = edge_positions(target_clear, grid.pixel_size)

summary = {
    "target_cd_nm": target_cd * 1e9,
    "l0_cd_nm": critical_dimension(printed_l0[cy, :], wafer.pixel_x_m) * 1e9,
    "l1_cd_nm": critical_dimension(printed_l1[cy, :], wafer.pixel_x_m) * 1e9,
    "l0_epe_nm": mean_absolute_epe(target_edges, edge_positions(printed_l0[cy, :], wafer.pixel_x_m)) * 1e9,
    "l1_epe_nm": mean_absolute_epe(target_edges, edge_positions(printed_l1[cy, :], wafer.pixel_x_m)) * 1e9,
    "l1_transition_width_nm": transition_width(gaussian_blur(aerial[cy, :], wafer.pixel_x_m, 2e-9), wafer.pixel_x_m) * 1e9,
}
summary

In [ ]:
sweep = blur_dose_sweep(
    aerial[cy, :],
    target_clear,
    wafer.pixel_x_m,
    sigma_values_m=[1e-9, 2e-9, 4e-9],
    doses=[0.8, 1.0, 1.2],
    threshold=0.2,
)

[
    {
        "sigma_nm": point.sigma_m * 1e9,
        "dose": point.dose,
        "cd_nm": point.cd_m * 1e9,
        "epe_nm": None if point.mean_abs_epe_m is None else point.mean_abs_epe_m * 1e9,
        "transition_width_nm": point.transition_width_m * 1e9,
        "lwr_proxy_nm": point.lwr_proxy_m * 1e9,
    }
    for point in sweep
]

In [ ]:
depth_result = focus_depth_resolved_resist(
    mask,
    grid,
    depth_values_m=[0.0, 40e-9, 80e-9],
    pupil_spec=PupilSpec(grid_size=512, na=C.NA_HIGH, obscuration_ratio=0.0),
    dose=1.0,
    threshold=0.2,
    absorption_coefficient_m_inv=6.0e6,
    anamorphic=False,
)

profile = sidewall_angle_proxy(
    depth_result.exposed_stack,
    depth_result.depth_values_m,
    depth_result.wafer.pixel_x_m,
)

{
    "top_bottom_asymmetry": top_bottom_dose_asymmetry(depth_result.dose_stack),
    "sidewall_angle_deg": profile.sidewall_angle_deg,
    "cd_delta_nm": profile.cd_delta_m * 1e9,
    "depth_profile": [
        {
            "depth_nm": item.depth_m * 1e9,
            "cd_nm": item.cd_m * 1e9,
            "exposed_fraction": item.exposed_fraction,
        }
        for item in profile.profile
    ],
    "slices": [
        {
            "depth_nm": item.depth_m * 1e9,
            "defocus_nm": item.defocus_m * 1e9,
            "attenuation": item.attenuation,
            "mean_dose": item.mean_dose,
            "exposed_fraction": item.exposed_fraction,
        }
        for item in depth_result.slice_summaries
    ],
}

In [ ]:
stochastic_curve = monte_carlo_lwr_curve(
    aerial[cy, :],
    wafer.pixel_x_m,
    doses=[0.8, 1.0, 1.2],
    trials=32,
    params=StochasticResistParams(
        photon_density_per_unit=500.0,
        material_threshold_sigma=0.01,
    ),
    seed=20260426,
)

[
    {
        "dose": point.dose,
        "mean_cd_nm": point.mean_cd_m * 1e9,
        "lwr_3sigma_nm": point.lwr_m * 1e9,
        "lcdu_3sigma_nm": point.lcdu_m * 1e9,
        "missing_fraction": point.missing_fraction,
        "budget_total_lwr_nm": point.budget.total_lwr_m * 1e9,
        "budget_optical_lwr_nm": point.budget.optical_lwr_m * 1e9,
        "budget_material_lwr_nm": point.budget.material_lwr_m * 1e9,
    }
    for point in stochastic_curve
]

In [ ]:
convergence = monte_carlo_convergence_gate(
    aerial[cy, :],
    wafer.pixel_x_m,
    dose=1.0,
    trial_counts=[100, 300, 1000],
    tolerance_fraction=0.10,
    params=StochasticResistParams(
        photon_density_per_unit=800.0,
        material_threshold_sigma=0.005,
    ),
    seed=1,
)

{
    "converged": convergence.converged,
    "relative_lwr_change": convergence.relative_lwr_change,
    "tolerance_fraction": convergence.tolerance_fraction,
    "final_lwr_nm": convergence.final_lwr_m * 1e9,
    "trial_lwr_nm": [
        {
            "trials": point.trials,
            "lwr_3sigma_nm": point.lwr_m * 1e9,
            "mean_cd_nm": point.mean_cd_m * 1e9,
        }
        for point in convergence.trial_results
    ],
}

In [ ]:
blur_calibration = calibrate_blur_lwr_proxy(
    sweep,
    [1.25 * point.lwr_proxy_m for point in sweep],
)
swa_calibration = calibrate_sidewall_angle_proxy(
    [profile],
    [profile.sidewall_angle_deg],
)
stochastic_calibration = calibrate_stochastic_lwr_budget(
    [point.dose for point in stochastic_curve],
    [max(point.mean_cd_m, wafer.pixel_x_m) for point in stochastic_curve],
    [point.budget.total_lwr_m for point in stochastic_curve],
    optical_coeff_grid_m=[3.0e-9],
    material_coeff_grid_m=[3.0e-9],
)

{
    "blur_lwr_scale": blur_calibration.scale,
    "blur_lwr_rms_nm": blur_calibration.rms_error_m * 1e9,
    "swa_slope": swa_calibration.slope,
    "swa_intercept_deg": swa_calibration.intercept_deg,
    "stochastic_optical_coeff_nm": stochastic_calibration.params.optical_lwr_coeff_m * 1e9,
    "stochastic_material_coeff_nm": stochastic_calibration.params.material_lwr_coeff_m * 1e9,
    "stochastic_lwr_rms_nm": stochastic_calibration.rms_error_m * 1e9,
}